In [ ]:
# ============================================================
# Core imports + config
# ============================================================
import os, sys, gc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from scipy.ndimage import gaussian_filter
from scipy.integrate import trapezoid
import seaborn as sns
import pickle
from tqdm.auto import tqdm

dt = 0.005
t_pre, t_post = 0.3, 0.3

SESSION_TYPE = "playback"  # "playback" or "eTheremin" session type for load_pickled_ss

USE_MACRO_EPOCH = True   # True = contiguous Condition blocks, False = peri-event windows
CEBRA_DISTANCE = "euclidean"  # "cosine" or "euclidean"
CEBRA_ARCH = "offset10-model" if CEBRA_DISTANCE == "cosine" else "offset10-model-mse"
TAU_SHIFT = 6       # label lag in bins (6 = 30ms at dt=0.005, 0 = disabled)
LIE_METHOD = "pytorch"  # "pytorch" = constrained skew-symmetric, "lstsq" = OLS + truncate
MIN_EPOCH_DUR = 2.0      # minimum macro-epoch duration (seconds)

NAS = r"\\129.199.81.18\data5\eTheremin"

mpl.rcdefaults()
plt.rcParams.update({
    'font.size': 7, 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
print("Imports ready.")

In [ ]:
# ============================================================
# Load spike-sorted Skieur data
# ============================================================

def load_pickled_ss(file_prefix, session_type, dt):
    data_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_data_ss")
    feat_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_feature_ss")
    with open(data_path, "rb") as f: n_data = pickle.load(f)
    with open(feat_path, "rb") as f: f_data = pickle.load(f)
    return n_data, f_data

print("Loading hs0...")
n_data_hs0, f_data_hs0 = load_pickled_ss("SKIEUR_hs_0", SESSION_TYPE, dt)
print(f"  hs0: {len(n_data_hs0)} sessions")

print("Loading hs1...")
n_data_hs1, f_data_hs1 = load_pickled_ss("SKIEUR_hs_1", SESSION_TYPE, dt)
print(f"  hs1: {len(n_data_hs1)} sessions")

# Add Velocity_x
for f_df in f_data_hs0 + f_data_hs1:
    pos = f_df["Position"].values
    vel = np.diff(pos); vel = np.append(0, vel)
    vel = vel * 100; vel[~np.isfinite(vel)] = 0
    f_df["Velocity_x"] = vel

n_data_all_raw = list(n_data_hs0) + list(n_data_hs1)
f_data_all_raw = list(f_data_hs0) + list(f_data_hs1)
n_hs0 = len(n_data_hs0)

MIN_GC = 10
n_data_all, f_data_all = [], []
n_hs0_filtered = 0
for i, nd in enumerate(n_data_all_raw):
    if nd.shape[0] >= MIN_GC:
        n_data_all.append(nd)
        f_data_all.append(f_data_all_raw[i])
        if i < n_hs0: n_hs0_filtered += 1

n_hs0 = n_hs0_filtered
print(f"After gc>={MIN_GC}: {len(n_data_all)} sessions (hs0={n_hs0}, hs1={len(n_data_all)-n_hs0})")

example = f_data_all[0]
print(f"Example: {example.shape[0]:,} tp, {n_data_all[0].shape[0]} neurons")
print(f"  Velocity_x: [{example['Velocity_x'].min():.1f}, {example['Velocity_x'].max():.1f}]")
print(f"  Position:   [{example['Position'].min():.1f}, {example['Position'].max():.1f}]")

In [ ]:
# ============================================================
# Helper functions: macro-epoch + unified extraction
# ============================================================


def preprocess_data(data_list, method="l2"):
    """Preprocess list of (time, neurons) arrays.
    method='l2': per-timepoint L2 normalization (for cosine distance)
    method='zscore': per-neuron Z-score across time (for euclidean distance)
    """
    out = []
    for d in data_list:
        if method == "zscore":
            mean = np.mean(d, axis=0, keepdims=True)
            std = np.std(d, axis=0, keepdims=True)
            std[std == 0] = 1e-9
            out.append(((d - mean) / std).astype(np.float32))
        else:  # l2
            norms = np.linalg.norm(d, axis=1, keepdims=True)
            norms[norms == 0] = 1e-9
            out.append((d / norms).astype(np.float32))
    return out


def extract_macro_epochs(n_data_session, f_df, condition_val, dt,
                          min_duration=2.0, label_col="Velocity_x"):
    """Extract contiguous macro-epochs of the same Condition."""
    conditions = f_df["Condition"].values
    mask = (conditions == condition_val)
    min_bins = int(min_duration / dt)
    epochs_n, epochs_l = [], []
    in_epoch, start = False, 0
    for i in range(len(mask)):
        if mask[i] and not in_epoch:
            start = i; in_epoch = True
        elif not mask[i] and in_epoch:
            if i - start >= min_bins:
                epochs_n.append(n_data_session[:, start:i].T.astype(np.float32))
                epochs_l.append(f_df[label_col].values[start:i].astype(np.float32))
            in_epoch = False
    if in_epoch and (len(mask) - start) >= min_bins:
        epochs_n.append(n_data_session[:, start:].T.astype(np.float32))
        epochs_l.append(f_df[label_col].values[start:].astype(np.float32))
    return epochs_n, epochs_l, len(epochs_n)


def extract_epochs(n_data_session, f_df, condition_val, dt,
                   label_col="Velocity_x", step=1):
    """Unified extraction: macro-epochs or peri-event windows."""
    if USE_MACRO_EPOCH:
        epochs_n, epochs_l, n_ep = extract_macro_epochs(
            n_data_session, f_df, condition_val, dt,
            min_duration=MIN_EPOCH_DUR, label_col=label_col)
        if step > 1:
            epochs_n = [e[::step].astype(np.float32) for e in epochs_n]
            epochs_l = [l[::step].astype(np.float32) for l in epochs_l]
        method = "l2" if CEBRA_DISTANCE == "cosine" else "zscore"
        epochs_n = preprocess_data(epochs_n, method=method)
        if TAU_SHIFT > 0:
            epochs_n, epochs_l = zip(*[(e_n[TAU_SHIFT:], e_l[:-TAU_SHIFT])
                                        for e_n, e_l in zip(epochs_n, epochs_l)])
            epochs_n, epochs_l = list(epochs_n), list(epochs_l)
        return epochs_n, epochs_l, n_ep
    else:
        trigger_mask = (f_df["Condition"].values == condition_val) & (f_df["Frequency_changes"].values == 1)
        trigger_indices = np.where(trigger_mask)[0]
        n_pre = int(t_pre / dt)
        n_post = int(t_post / dt)
        windows_n, windows_l = [], []
        for idx in trigger_indices:
            start = idx - n_pre
            end = idx + n_post + 1
            if start >= 0 and end <= n_data_session.shape[1]:
                windows_n.append(n_data_session[:, start:end].T.astype(np.float32))
                windows_l.append(f_df[label_col].values[start:end].astype(np.float32))
        if step > 1:
            windows_n = [w[::step].astype(np.float32) for w in windows_n]
            windows_l = [l[::step].astype(np.float32) for l in windows_l]
        method = "l2" if CEBRA_DISTANCE == "cosine" else "zscore"
        windows_n = preprocess_data(windows_n, method=method)
        if TAU_SHIFT > 0:
            windows_n, windows_l = zip(*[(w_n[TAU_SHIFT:], w_l[:-TAU_SHIFT])
                                          for w_n, w_l in zip(windows_n, windows_l)])
            windows_n, windows_l = list(windows_n), list(windows_l)
        return windows_n, windows_l, len(windows_n)



print("Helpers ready: preprocess_data, extract_macro_epochs, extract_epochs")


In [ ]:
# ============================================================
# Macro-Epoch / Peri-Event Extraction Report
# Runs extract_epochs() for all sessions and prints counts.
# Use this to check epoch/trigger counts before heavy computation.
# ============================================================
for label_col, driver_name in [("Velocity_x", "Velocity"), ("Position", "Position")]:
    print("=" * 50)
    print(f"  {driver_name}-Driven")
    print("=" * 50)
    for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
        total_epochs, total_pts, total_sec = 0, 0, 0
        for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
            epochs_n, epochs_l, n_ep = extract_epochs(
                n_data_session, f_df, val, dt, label_col=label_col)
            total_epochs += n_ep
            total_pts += sum(e.shape[0] for e in epochs_n)
            total_sec += sum(e.shape[0] for e in epochs_n) * dt
        mode = "macro-epoch" if USE_MACRO_EPOCH else "peri-event"
        print(f"  {cond_name}: {total_epochs} {mode}s, {total_pts:,} pts, {total_sec:.0f}s total")
    print()

print("Proceed to analysis cells.")